# 🏊 SwimCoach RAG System
### Azure OpenAI + Pinecone + Azure Cognitive Search
**Assignment 1 – Generative AI | Professional Swimming Coach RAG Solution**

---
**Architecture:**
```
VIDEO → ffmpeg chunks → OpenCV frames → GPT-4o Vision captions
      → GPT-4o summarise → Ada-002 embed → Pinecone + ACS + Cosmos DB

QUERY → GPT-4o decompose → Parallel (BM25 + Semantic) → RRF Fusion
      → GPT-4o Reranker → GPT-4 Turbo SwimCoachGPT → Streamed response
```

> **Run cells top to bottom.** Fill in your API keys in Cell 2 before proceeding.

## 📦 Cell 1 — Install Dependencies

In [ ]:
%%capture
!pip install openai pinecone-client azure-search-documents azure-cosmos \
            azure-storage-blob opencv-python-headless ffmpeg-python \
            tenacity rich tqdm python-dotenv
print("✅ All packages installed")

## 🔑 Cell 2 — API Keys & Configuration
> **Fill in every value below before running any other cell.**  
> All keys are stored only in this Colab session — nothing is saved to disk.

In [ ]:
import os

# ── Azure OpenAI ──────────────────────────────────────────────────────────────
os.environ["AZURE_OPENAI_API_KEY"]               = "YOUR_AZURE_OPENAI_API_KEY"
os.environ["AZURE_OPENAI_ENDPOINT"]              = "https://YOUR-RESOURCE.openai.azure.com/"
os.environ["AZURE_OPENAI_API_VERSION"]           = "2024-02-15-preview"
os.environ["AZURE_OPENAI_GPT4O_DEPLOYMENT"]      = "gpt-4o"          # your deployment name
os.environ["AZURE_OPENAI_GPT4_TURBO_DEPLOYMENT"] = "gpt-4-turbo"     # your deployment name
os.environ["AZURE_OPENAI_ADA_DEPLOYMENT"]        = "text-embedding-ada-002"

# ── Pinecone ──────────────────────────────────────────────────────────────────
os.environ["PINECONE_API_KEY"]    = "YOUR_PINECONE_API_KEY"
os.environ["PINECONE_INDEX_NAME"] = "swimcoach-kb"
os.environ["PINECONE_CLOUD"]      = "aws"
os.environ["PINECONE_REGION"]     = "us-east-1"

# ── Azure Cognitive Search (BM25) ─────────────────────────────────────────────
os.environ["AZURE_SEARCH_ENDPOINT"]   = "https://YOUR-SEARCH.search.windows.net"
os.environ["AZURE_SEARCH_API_KEY"]    = "YOUR_SEARCH_API_KEY"
os.environ["AZURE_SEARCH_INDEX_NAME"] = "swimcoach-bm25"

# ── Azure Cosmos DB (optional — metadata store) ───────────────────────────────
os.environ["COSMOS_ENDPOINT"]  = "https://YOUR-COSMOS.documents.azure.com:443/"
os.environ["COSMOS_KEY"]       = "YOUR_COSMOS_KEY"
os.environ["COSMOS_DATABASE"]  = "swimcoach"
os.environ["COSMOS_CONTAINER"] = "coaching_logs"

print("✅ Configuration set")

## ⚙️ Cell 3 — Initialise Azure / Pinecone Clients

In [ ]:
from openai import AzureOpenAI
from pinecone import Pinecone, ServerlessSpec
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchFieldDataType,
)
from azure.core.credentials import AzureKeyCredential

# Azure OpenAI client
openai_client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
)
GPT4O_DEPLOYMENT      = os.environ["AZURE_OPENAI_GPT4O_DEPLOYMENT"]
GPT4_TURBO_DEPLOYMENT = os.environ["AZURE_OPENAI_GPT4_TURBO_DEPLOYMENT"]
ADA_DEPLOYMENT        = os.environ["AZURE_OPENAI_ADA_DEPLOYMENT"]

# Pinecone
_pc                 = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
PINECONE_INDEX_NAME = os.environ["PINECONE_INDEX_NAME"]

def get_pinecone_index():
    existing = [i.name for i in _pc.list_indexes()]
    if PINECONE_INDEX_NAME not in existing:
        _pc.create_index(
            name=PINECONE_INDEX_NAME,
            dimension=1536,
            metric="cosine",
            spec=ServerlessSpec(
                cloud=os.environ["PINECONE_CLOUD"],
                region=os.environ["PINECONE_REGION"],
            ),
        )
        print(f"[Pinecone] Created index '{PINECONE_INDEX_NAME}'")
    return _pc.Index(PINECONE_INDEX_NAME)

# Azure Cognitive Search
SEARCH_ENDPOINT   = os.environ["AZURE_SEARCH_ENDPOINT"]
SEARCH_API_KEY    = os.environ["AZURE_SEARCH_API_KEY"]
SEARCH_INDEX_NAME = os.environ["AZURE_SEARCH_INDEX_NAME"]
_search_cred      = AzureKeyCredential(SEARCH_API_KEY)

def get_search_client():
    return SearchClient(SEARCH_ENDPOINT, SEARCH_INDEX_NAME, _search_cred)

def ensure_search_index():
    idx_client = SearchIndexClient(SEARCH_ENDPOINT, _search_cred)
    existing = [i.name for i in idx_client.list_indexes()]
    if SEARCH_INDEX_NAME in existing:
        return
    fields = [
        SimpleField(name="id",             type=SearchFieldDataType.String, key=True),
        SimpleField(name="video_id",       type=SearchFieldDataType.String, filterable=True),
        SimpleField(name="doc_type",       type=SearchFieldDataType.String, filterable=True),
        SimpleField(name="difficulty",     type=SearchFieldDataType.String, filterable=True),
        SearchableField(name="embedding_text",  type=SearchFieldDataType.String, analyzer_name="en.microsoft"),
        SearchableField(name="topic_tags_str",  type=SearchFieldDataType.String),
    ]
    idx_client.create_index(SearchIndex(name=SEARCH_INDEX_NAME, fields=fields))
    print(f"[ACS] Created BM25 index '{SEARCH_INDEX_NAME}'")

# Cosmos DB (optional)
def get_cosmos_container():
    from azure.cosmos import CosmosClient
    client = CosmosClient(os.environ["COSMOS_ENDPOINT"], os.environ["COSMOS_KEY"])
    db = client.get_database_client(os.environ["COSMOS_DATABASE"])
    return db.get_container_client(os.environ["COSMOS_CONTAINER"])

print("✅ Clients initialised")

## 🛠️ Cell 4 — Shared Helpers (Embed, Caption, Summarise)

In [ ]:
import base64, time
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from openai import RateLimitError, APIError

# ── Ada-002 embedding ─────────────────────────────────────────────────────────
@retry(retry=retry_if_exception_type((RateLimitError, APIError)),
       wait=wait_exponential(min=2, max=30), stop=stop_after_attempt(5))
def embed_text(text: str) -> list:
    r = openai_client.embeddings.create(model=ADA_DEPLOYMENT, input=text)
    return r.data[0].embedding

# ── GPT-4o Vision frame caption ───────────────────────────────────────────────
FRAME_CAPTION_SYSTEM = """You are an expert swimming coach and video analyst.
Caption each video frame in 1-3 sentences. Focus on:
- Stroke phase (catch, pull, push, recovery)
- Body position (head, hips, arms, legs alignment)
- Kick mechanics (frequency, amplitude, dolphin kick depth)
Be precise and use technical swimming terminology."""

@retry(retry=retry_if_exception_type((RateLimitError, APIError)),
       wait=wait_exponential(min=2, max=60), stop=stop_after_attempt(5))
def caption_frame_b64(b64_image: str) -> str:
    r = openai_client.chat.completions.create(
        model=GPT4O_DEPLOYMENT,
        messages=[
            {"role": "system", "content": FRAME_CAPTION_SYSTEM},
            {"role": "user", "content": [
                {"type": "image_url",
                 "image_url": {"url": f"data:image/jpeg;base64,{b64_image}", "detail": "low"}},
                {"type": "text", "text": "Describe this swimming video frame for coaching analysis."},
            ]},
        ],
        max_tokens=200, temperature=0.2,
    )
    return r.choices[0].message.content.strip()

# ── GPT-4o summarise captions → embedding_text ───────────────────────────────
SUMMARY_SYSTEM = """You are a swimming coaching knowledge-base curator.
Synthesise per-frame captions into a single 10-200 word summary capturing:
- Primary technique or drill demonstrated
- Camera angle and underwater perspective
- Skill level (beginner / intermediate / advanced / elite)
- Key coaching focus points
Be dense with domain-specific keywords (stroke name, kick type, phase names)."""

@retry(retry=retry_if_exception_type((RateLimitError, APIError)),
       wait=wait_exponential(min=2, max=60), stop=stop_after_attempt(5))
def summarise_captions(captions: list) -> str:
    joined = "\n".join(f"- {c}" for c in captions)
    r = openai_client.chat.completions.create(
        model=GPT4O_DEPLOYMENT,
        messages=[
            {"role": "system", "content": SUMMARY_SYSTEM},
            {"role": "user",   "content": f"Frame captions:\n{joined}\n\nWrite the final video summary:"},
        ],
        max_tokens=300, temperature=0.2,
    )
    return r.choices[0].message.content.strip()

print("✅ Helper functions ready")

## 📚 Cell 5 — Knowledge Base Seed Data
*8 video catalog entries + 5 coaching logs + 1 athlete profile matching the assignment §2.2 schema.*

In [ ]:
VIDEO_CATALOG = [
    {"video_id":"V-0091","doc_type":"video_catalog","title":"Elite Butterfly Breakout & Dolphin Kick",
     "topic_tags":["butterfly","dolphin kick","breakout","underwater","advanced"],"difficulty":"advanced",
     "embedding_text":"Elite butterfly breakout technique filmed from underwater side angle. Demonstrates hip-driven dolphin kick with high amplitude and minimal drag. Athlete achieves 7.5m underwater breakout distance post-turn. Focus on tight streamline position, undulation initiation from core, and timing of first stroke pull relative to surfacing. Advanced drill for competitive butterfly swimmers targeting sub-2:00 200m."},
    {"video_id":"V-0103","doc_type":"video_catalog","title":"Hip Drive & Body Undulation for Butterfly",
     "topic_tags":["butterfly","hip rotation","body undulation","intermediate"],"difficulty":"intermediate",
     "embedding_text":"Poolside side-angle footage demonstrating hip drive mechanics for butterfly stroke. Slow-motion segments highlight the two-beat kick rhythm synchronized with arm entry. Body undulation sourced from hips and core, not shoulders. Prerequisite: basic butterfly arm cycle mastery. Intermediate level."},
    {"video_id":"V-0017","doc_type":"video_catalog","title":"Freestyle Flip Turn — Timing & Underwater Streamline",
     "topic_tags":["freestyle","flip turn","tumble turn","underwater streamline","beginner"],"difficulty":"beginner",
     "embedding_text":"Underwater end-wall camera captures freestyle flip turn from approach through push-off. Highlights: touch-to-flip timing 0.8s, tight tuck radius, wall contact at 0.45m depth, streamline lock before dolphin kicks. Common error corrections included. Suitable for swimmers new to competitive flip turn technique."},
    {"video_id":"V-0022","doc_type":"video_catalog","title":"Underwater Dolphin Kick Extension Post-Turn (Freestyle)",
     "topic_tags":["freestyle","dolphin kick","underwater","post-turn","intermediate"],"difficulty":"intermediate",
     "embedding_text":"Side-angle underwater footage of elite swimmer executing 7-8m dolphin kick sequence after freestyle flip turn. Body depth maintained at 0.5m for reduced drag. Kick amplitude narrows progressively toward surface. Timing cue for breakout pull shown. Intermediate drill; assumes solid flip turn mechanics."},
    {"video_id":"V-0044","doc_type":"video_catalog","title":"Butterfly Underwater Pull Phase — Catch to Finish",
     "topic_tags":["butterfly","pull phase","catch","underwater","advanced"],"difficulty":"advanced",
     "embedding_text":"Frontal underwater angle isolating butterfly pull phase from high-elbow catch through in-sweep and finish. Pressure points marked with coaching overlays. Common faults: dropped elbow, early breathing disrupting pull line. Advanced video; best viewed after V-0103 hip-drive foundation."},
    {"video_id":"V-0067","doc_type":"video_catalog","title":"Backstroke Catch Mechanics — High-Elbow Entry",
     "topic_tags":["backstroke","catch mechanics","high elbow","shoulder rotation","intermediate"],"difficulty":"intermediate",
     "embedding_text":"Overhead aerial camera shows backstroke high-elbow catch from hand entry to catch lock. Shoulder rotation 45-55 degrees per stroke cycle measured. Drill progressions: single-arm, fingertip drag, full stroke with tempo trainer. Targets swimmers with dropped-wrist catch fault. Intermediate difficulty."},
    {"video_id":"V-0071","doc_type":"video_catalog","title":"Backstroke Shoulder Roll & Body Rotation",
     "topic_tags":["backstroke","shoulder rotation","body roll","beginner"],"difficulty":"beginner",
     "embedding_text":"Side-poolside footage demonstrating optimal shoulder and hip rotation for backstroke. 60-degree body roll shown in slow motion. Connection between rotation and pull power explained. Beginner-friendly with drill progressions using kickboard for isolation."},
    {"video_id":"V-0115","doc_type":"video_catalog","title":"Race Strategy — 200m Butterfly Split Pacing",
     "topic_tags":["butterfly","race strategy","pacing","split times","mental conditioning"],"difficulty":"intermediate",
     "embedding_text":"Race analysis video breaking down 200m butterfly split pacing strategy. Optimal split ratios: 25-26/29-30/31-32/30-31 seconds per 50m. Common mistake: going out too fast on first 50m causing catastrophic fade on final 50m. Includes elite race footage comparison and stroke rate data. Intermediate mental conditioning content."},
]

COACHING_LOGS = [
    {"log_id":"SWM-2024-0312","doc_type":"coaching_log",
     "athlete_profile":{"age":19,"gender":"M","event":"200m Butterfly","weakness":["dolphin kick"]},
     "videos_recommended":[{"video_id":"V-0091","watch_pct":92},{"video_id":"V-0044","watch_pct":78}],
     "performance_delta":{"before_s":121.4,"after_s":119.8,"improvement_pct":1.3},"weeks_to_competition":6,
     "notes":"Athlete showed marked improvement in breakout distance after V-0091 series."},
    {"log_id":"SWM-2024-0289","doc_type":"coaching_log",
     "athlete_profile":{"age":21,"gender":"M","event":"200m Butterfly","weakness":["dolphin kick","hip rotation"]},
     "videos_recommended":[{"video_id":"V-0091","watch_pct":88},{"video_id":"V-0103","watch_pct":95}],
     "performance_delta":{"before_s":118.2,"after_s":117.1,"improvement_pct":0.93},"weeks_to_competition":8,
     "notes":"V-0103 hip-drive drill combined with V-0091 produced compound improvement."},
    {"log_id":"SWM-2024-0401","doc_type":"coaching_log",
     "athlete_profile":{"age":18,"gender":"M","event":"100m Freestyle","weakness":["flip turn","underwater streamline"]},
     "videos_recommended":[{"video_id":"V-0017","watch_pct":97},{"video_id":"V-0022","watch_pct":84}],
     "performance_delta":{"before_s":52.3,"after_s":51.8,"improvement_pct":0.96},"weeks_to_competition":4,
     "notes":"Flip turn timing corrected; underwater dolphin kick extended to 7m."},
    {"log_id":"SWM-2024-0455","doc_type":"coaching_log",
     "athlete_profile":{"age":22,"gender":"F","event":"200m Backstroke","weakness":["catch mechanics","shoulder rotation"]},
     "videos_recommended":[{"video_id":"V-0067","watch_pct":91},{"video_id":"V-0071","watch_pct":73}],
     "performance_delta":{"before_s":133.5,"after_s":132.1,"improvement_pct":1.05},"weeks_to_competition":10,
     "notes":"Overhead camera angle in V-0067 proved critical for catch awareness."},
    {"log_id":"SWM-2024-0512","doc_type":"coaching_log",
     "athlete_profile":{"age":20,"gender":"M","event":"200m Butterfly","weakness":["race strategy","pacing"]},
     "videos_recommended":[{"video_id":"V-0091","watch_pct":80},{"video_id":"V-0115","watch_pct":90}],
     "performance_delta":{"before_s":120.0,"after_s":118.5,"improvement_pct":1.25},"weeks_to_competition":5,
     "notes":"Race-strategy video V-0115 helped athlete manage split pacing more effectively."},
]

ATHLETE_PROFILES = [
    {"athlete_id":"ATH-001","doc_type":"athlete_profile","age":19,"gender":"M",
     "primary_event":"200m Butterfly","pb_seconds":121.4,"weaknesses":["dolphin kick"],
     "videos_watched":["V-0103"],
     "embedding_text":"Male butterfly swimmer aged 19. Primary event 200m butterfly, PB 2:01.4. Identified weakness: dolphin kick depth and amplitude during breakout. Has watched hip-drive drill video. Needs advanced breakout and kick extension content. 6 weeks to State Championships."},
]

print(f"✅ Seed data loaded: {len(VIDEO_CATALOG)} videos, {len(COACHING_LOGS)} coaching logs, {len(ATHLETE_PROFILES)} athlete profiles")

## 🗄️ Cell 6 — Index Document Function
*Upserts one document into Pinecone (semantic) + Azure Cognitive Search (BM25) + Cosmos DB (metadata).*

In [ ]:
def index_document(doc: dict, embedding_text: str):
    doc_id = doc.get("video_id") or doc.get("log_id") or doc.get("athlete_id")

    # Embed
    vector = embed_text(embedding_text)

    # Pinecone
    pc_index = get_pinecone_index()
    pc_index.upsert(vectors=[{
        "id": doc_id,
        "values": vector,
        "metadata": {
            "doc_type":       doc.get("doc_type", ""),
            "video_id":       doc.get("video_id", ""),
            "topic_tags":     doc.get("topic_tags", []),
            "difficulty":     doc.get("difficulty", ""),
            "embedding_text": embedding_text[:500],
        },
    }])

    # Azure Cognitive Search
    ensure_search_index()
    get_search_client().upload_documents(documents=[{
        "id":             doc_id.replace("-", "_"),
        "video_id":       doc.get("video_id", ""),
        "doc_type":       doc.get("doc_type", ""),
        "difficulty":     doc.get("difficulty", ""),
        "embedding_text": embedding_text,
        "topic_tags_str": " ".join(doc.get("topic_tags", [])),
    }])

    # Cosmos DB (optional)
    try:
        container = get_cosmos_container()
        container.upsert_item({**doc, "id": doc_id, "embedding_text": embedding_text})
    except Exception as e:
        pass  # Cosmos is optional

    print(f"  ✅ Indexed: {doc_id}")

print("✅ index_document() ready")

## 🌱 Cell 7 — Seed the Knowledge Base
*Indexes all seed documents into Pinecone + ACS. Run once. Safe to re-run (upsert).*

In [ ]:
from tqdm.notebook import tqdm

all_docs = []

# Video catalog — embedding_text is pre-written (no video needed)
for v in VIDEO_CATALOG:
    all_docs.append((v, v["embedding_text"]))

# Coaching logs — build embedding_text from structured fields
for log in COACHING_LOGS:
    ap = log["athlete_profile"]
    et = (
        f"Coaching log {log['log_id']}. "
        f"Athlete: {ap['age']}yr {ap['gender']} {ap['event']}. "
        f"Weaknesses: {', '.join(ap['weakness'])}. "
        f"Videos recommended: {', '.join(v['video_id'] for v in log['videos_recommended'])}. "
        f"Performance improvement: {log['performance_delta']['improvement_pct']}%. "
        f"Notes: {log.get('notes', '')}"
    )
    all_docs.append((log, et))

# Athlete profiles
for ath in ATHLETE_PROFILES:
    all_docs.append((ath, ath["embedding_text"]))

print(f"Indexing {len(all_docs)} documents into Pinecone + ACS + Cosmos DB ...\n")
for doc, et in tqdm(all_docs, desc="Indexing"):
    try:
        index_document(doc, et)
    except Exception as e:
        print(f"  ❌ Error: {e}")

print(f"\n✅ Knowledge base seeded with {len(all_docs)} documents")

## 🎬 Cell 8 — Data Ingestion Pipeline (Real Video)
*Upload a `.mp4 / .mov / .avi` file to Colab, then run this cell.*  
**Stages:** ffmpeg chunks → OpenCV frames → GPT-4o Vision captions → GPT-4o summary → Ada-002 embed → Index

> Skip this cell if you only want to test the query pipeline with seed data.

In [ ]:
import subprocess, tempfile, cv2, base64
from pathlib import Path
from tqdm.notebook import tqdm

CHUNK_SIZE_MB   = 20
FRAME_INTERVAL  = 2.5   # seconds between sampled frames
MAX_FRAMES_CHUNK = 40

# ── ffmpeg: split into chunks ─────────────────────────────────────────────────
def chunk_video(video_path, output_dir):
    import ffmpeg as ffmpeg_lib
    video_path = Path(video_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    file_size_mb = video_path.stat().st_size / (1024*1024)
    probe = ffmpeg_lib.probe(str(video_path))
    duration = float(probe["format"]["duration"])
    n_chunks = max(1, int(file_size_mb // CHUNK_SIZE_MB) + 1)
    segment_time = duration / n_chunks
    pattern = str(output_dir / "chunk_%03d.mp4")
    subprocess.run([
        "ffmpeg", "-y", "-i", str(video_path), "-c", "copy",
        "-f", "segment", "-segment_time", str(segment_time),
        "-reset_timestamps", "1", pattern,
    ], check=True, capture_output=True)
    chunks = sorted(output_dir.glob("chunk_*.mp4"))
    print(f"  📦 {len(chunks)} chunks from {file_size_mb:.1f} MB video")
    return [str(c) for c in chunks]

# ── OpenCV: extract frames ────────────────────────────────────────────────────
def extract_frames(chunk_path, output_dir):
    cap = cv2.VideoCapture(chunk_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    skip = int(fps * FRAME_INTERVAL)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    frame_paths, frame_idx, saved = [], 0, 0
    while saved < MAX_FRAMES_CHUNK:
        ret, frame = cap.read()
        if not ret: break
        if frame_idx % skip == 0:
            out_path = str(output_dir / f"frame_{saved:04d}.jpg")
            cv2.imwrite(out_path, frame, [cv2.IMWRITE_JPEG_QUALITY, 85])
            frame_paths.append(out_path)
            saved += 1
        frame_idx += 1
    cap.release()
    return frame_paths

# ── Full pipeline ─────────────────────────────────────────────────────────────
def ingest_video(video_path, video_id, topic_tags=None, difficulty="intermediate"):
    print(f"\n🎬 Ingesting: {video_path} → {video_id}")
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)

        # Stage 1: chunk
        print("Stage 1: Chunking with ffmpeg ...")
        chunks = chunk_video(video_path, tmp / "chunks")

        # Stages 2+3: extract frames & caption
        all_captions = []
        for i, chunk in enumerate(chunks):
            print(f"Stage 2-3: Chunk {i+1}/{len(chunks)} — extracting & captioning frames ...")
            frames = extract_frames(chunk, tmp / f"frames_{i}")
            print(f"  {len(frames)} frames extracted")
            for fp in tqdm(frames, desc="  GPT-4o Vision", unit="frame"):
                with open(fp, "rb") as f:
                    b64 = base64.b64encode(f.read()).decode()
                all_captions.append(caption_frame_b64(b64))

        # Stage 4: summarise
        print("Stage 4: Summarising captions with GPT-4o ...")
        embedding_text = summarise_captions(all_captions)
        print(f"  embedding_text ({len(embedding_text.split())} words):")
        print(f"  {embedding_text}\n")

        # Stage 5: index
        print("Stage 5: Embedding + indexing ...")
        doc = {"video_id": video_id, "doc_type": "video_catalog",
               "topic_tags": topic_tags or [], "difficulty": difficulty,
               "embedding_text": embedding_text}
        index_document(doc, embedding_text)

    print(f"\n✅ Ingestion complete for {video_id}")

# ─────────────────────────────────────────────────────────────────────────────
# To ingest a video, upload it to Colab then call:
#
# ingest_video(
#     video_path  = "/content/your_video.mp4",
#     video_id    = "V-0200",
#     topic_tags  = ["butterfly", "dolphin kick"],
#     difficulty  = "advanced",
# )
# ─────────────────────────────────────────────────────────────────────────────
print("✅ ingest_video() ready — see comment above to run")

## 🔍 Cell 9 — RAG Query Pipeline
*All 5 stages: Decompose → Parallel Retrieve → RRF Fusion → LLM Rerank → SwimCoachGPT*

In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor

# ── Domain guard ──────────────────────────────────────────────────────────────
SWIM_KEYWORDS = {
    "swim","butterfly","freestyle","backstroke","breaststroke","medley",
    "dolphin","stroke","kick","athlete","coach","video","drill","technique",
    "race","competition","championship","pool","turn","breakout","split",
    "pacing","streamline","catch","pull","recovery","underwater","flip turn",
}
OUT_OF_DOMAIN = ("Sorry, I'm unable to give the information. "
                 "Please re-phrase your query with respect to my domain knowledge.")

def is_in_domain(q): return any(kw in q.lower() for kw in SWIM_KEYWORDS)

# ── Stage 1: Query Decomposer ─────────────────────────────────────────────────
DECOMPOSE_SYSTEM = """You are a swimming coaching assistant query parser.
Extract structured filters from the coach's natural language query.
Return ONLY valid JSON with these fields (use null if unknown):
{"event":"e.g. 200m Butterfly","weakness_tags":["dolphin kick"],
 "age_range":[18,22],"gender":"M"|"F"|null,"weeks_to_competition":6,
 "videos_already_watched":["V-0103"],"intent":"gap_analysis"|"sequence"|"improvement"|"general",
 "difficulty_filter":"beginner"|"intermediate"|"advanced"|null}"""

def decompose_query(query):
    r = openai_client.chat.completions.create(
        model=GPT4O_DEPLOYMENT,
        messages=[{"role":"system","content":DECOMPOSE_SYSTEM},
                  {"role":"user","content":query}],
        response_format={"type":"json_object"}, max_tokens=400, temperature=0,
    )
    filters = json.loads(r.choices[0].message.content)
    print(f"  Filters extracted: {json.dumps(filters, indent=2)}")
    return filters

# ── Stage 2: Semantic search (Pinecone) ───────────────────────────────────────
def semantic_search(query, filters, top_k=10):
    vector = embed_text(query)
    pc_filter = {}
    if filters.get("difficulty_filter"):
        pc_filter["difficulty"] = {"$eq": filters["difficulty_filter"]}
    results = get_pinecone_index().query(
        vector=vector, top_k=top_k, include_metadata=True,
        filter=pc_filter if pc_filter else None,
    )
    return [{"id":m.id,"rank_semantic":i+1,"score_semantic":m.score,"metadata":m.metadata,"source":"semantic"}
            for i,m in enumerate(results.matches)]

# ── Stage 2: BM25 search (Azure Cognitive Search) ─────────────────────────────
def bm25_search(query, filters, top_k=10):
    odata_filter = None
    if filters.get("difficulty_filter"):
        odata_filter = f"difficulty eq '{filters['difficulty_filter']}'"
    results = get_search_client().search(
        search_text=query, top=top_k, filter=odata_filter,
        select=["id","video_id","doc_type","difficulty","embedding_text","topic_tags_str"],
    )
    return [{"id":r.get("video_id") or r["id"],"rank_bm25":i+1,"score_bm25":r.get("@search.score",0),
             "metadata":{"embedding_text":r.get("embedding_text",""),"doc_type":r.get("doc_type",""),
                         "difficulty":r.get("difficulty",""),"topic_tags":r.get("topic_tags_str","").split()},
             "source":"bm25"}
            for i,r in enumerate(results)]

# ── Stage 3: RRF Fusion (50/50, k=60) ────────────────────────────────────────
RRF_K = 60
def rrf_fusion(sem_hits, bm25_hits, top_n=20):
    scores = {}
    for h in sem_hits:
        scores.setdefault(h["id"], {"rrf":0.0,"metadata":h["metadata"]})
        scores[h["id"]]["rrf"] += 0.5 * (1 / (RRF_K + h["rank_semantic"]))
    for h in bm25_hits:
        scores.setdefault(h["id"], {"rrf":0.0,"metadata":h["metadata"]})
        scores[h["id"]]["rrf"] += 0.5 * (1 / (RRF_K + h["rank_bm25"]))
    merged = [{"id":k,**v} for k,v in scores.items()]
    merged.sort(key=lambda x: x["rrf"], reverse=True)
    return merged[:top_n]

# ── Stage 4: LLM Reranker (GPT-4o) ───────────────────────────────────────────
RERANKER_SYSTEM = """You are an expert swimming coach assistant acting as a reranker.
Score each candidate document 1-10 based on relevance to the query. Consider:
  - Athlete event match (butterfly vs freestyle etc.)
  - Weakness relevance (dolphin kick, catch mechanics etc.)
  - Age/experience appropriateness
  - Prerequisite ordering (beginner before advanced)
Return ONLY valid JSON: {"scores":[{"doc_id":"...","relevance_score":8,"reason":"..."}]}"""

def llm_rerank(query, candidates, filters, top_n=5):
    top10 = candidates[:10]
    candidate_text = "\n\n".join(
        f"doc_id: {c['id']}\ntype: {c['metadata'].get('doc_type','')}\n"
        f"difficulty: {c['metadata'].get('difficulty','')}\n"
        f"tags: {', '.join(c['metadata'].get('topic_tags',[]))}\n"
        f"summary: {c['metadata'].get('embedding_text','')[:300]}"
        for c in top10
    )
    watched = filters.get("videos_already_watched") or []
    context = (f"Coach query: {query}\nAthlete event: {filters.get('event')}\n"
               f"Weaknesses: {filters.get('weakness_tags')}\nAge range: {filters.get('age_range')}\n"
               f"Weeks to competition: {filters.get('weeks_to_competition')}\n"
               f"Already watched: {watched}\n\nCandidates:\n{candidate_text}")
    r = openai_client.chat.completions.create(
        model=GPT4O_DEPLOYMENT,
        messages=[{"role":"system","content":RERANKER_SYSTEM},
                  {"role":"user","content":context}],
        response_format={"type":"json_object"}, max_tokens=600, temperature=0,
    )
    parsed = json.loads(r.choices[0].message.content)
    scores = parsed.get("scores", parsed) if isinstance(parsed, dict) else parsed
    scores.sort(key=lambda x: x["relevance_score"], reverse=True)
    return scores[:top_n]

# ── Stage 5: SwimCoachGPT (GPT-4 Turbo) ──────────────────────────────────────
SWIMCOACH_SYSTEM = """You are SwimCoachGPT, an elite AI swimming coach assistant.
Provide structured, evidence-based coaching recommendations grounded ONLY in
the retrieved context. Always cite video IDs.

Format your response in EXACTLY these 5 sections:

## 1. Must-Watch Videos
List video IDs with rationale and athlete match score (1-10).

## 2. Videos to Avoid Now
List videos that are too advanced, need prerequisites, or are not yet relevant.

## 3. Gap Analysis
Videos watched by similar athletes that this athlete has NOT yet watched.

## 4. Suggested Viewing Sequence
Ordered playlist with rationale for each step.

## 5. Confidence Level
State HIGH / MEDIUM / LOW and explain based on how many matching historical cases exist.

If query is not swimming-related, respond: Sorry, I'm unable to give the information.
Please re-phrase your query with respect to my domain knowledge."""

def generate_response(query, reranked, candidates, filters):
    reranked_ids = {r["doc_id"] for r in reranked}
    context_docs = [c for c in candidates if c["id"] in reranked_ids] or candidates[:5]
    context_text = "\n\n---\n\n".join(
        f"[{c['id']}]\nType: {c['metadata'].get('doc_type','')}\n"
        f"Tags: {', '.join(c['metadata'].get('topic_tags',[]))}\n"
        f"Difficulty: {c['metadata'].get('difficulty','')}\n"
        f"Summary: {c['metadata'].get('embedding_text','')}"
        for c in context_docs
    )
    rerank_ctx = "\n".join(f"- {r['doc_id']} (score {r['relevance_score']}): {r['reason']}" for r in reranked)
    user_msg = (f"Coach query: {query}\n\nAthlete profile:\n{json.dumps(filters,indent=2)}\n\n"
                f"Reranker assessment:\n{rerank_ctx}\n\nKnowledge base context:\n{context_text}")
    r = openai_client.chat.completions.create(
        model=GPT4_TURBO_DEPLOYMENT,
        messages=[{"role":"system","content":SWIMCOACH_SYSTEM},
                  {"role":"user","content":user_msg}],
        max_tokens=1500, temperature=0.3,
    )
    return r.choices[0].message.content

# ── Full RAG query orchestrator ───────────────────────────────────────────────
def run_rag_query(query):
    print(f"\n{'='*65}")
    print(f"🏊 SwimCoach RAG Query Pipeline")
    print(f"{'='*65}")
    print(f"Query: {query}\n")

    if not is_in_domain(query):
        print(f"⚠️  {OUT_OF_DOMAIN}")
        return OUT_OF_DOMAIN

    print("Stage 1: Decomposing query ...")
    filters = decompose_query(query)

    print("\nStage 2: Parallel hybrid retrieval ...")
    with ThreadPoolExecutor(max_workers=2) as ex:
        fut_sem  = ex.submit(semantic_search, query, filters, 10)
        fut_bm25 = ex.submit(bm25_search,    query, filters, 10)
        sem_hits  = fut_sem.result()
        bm25_hits = fut_bm25.result()
    print(f"  Semantic: {len(sem_hits)} hits | BM25: {len(bm25_hits)} hits")

    print("\nStage 3: RRF Fusion ...")
    merged = rrf_fusion(sem_hits, bm25_hits, top_n=20)
    print(f"  Top-{len(merged)} merged docs by RRF score")

    print("\nStage 4: LLM Reranking (GPT-4o) ...")
    reranked = llm_rerank(query, merged, filters, top_n=5)
    for r in reranked:
        print(f"  {r['doc_id']} → score {r['relevance_score']}: {r['reason'][:70]}")

    print("\nStage 5: SwimCoachGPT generating response ...\n")
    response = generate_response(query, reranked, merged, filters)

    print(f"{'─'*65}")
    print("🤖 SwimCoachGPT Response\n")
    print(response)
    print(f"{'─'*65}\n")
    return response

print("✅ Full RAG query pipeline ready")

## 🎯 Cell 10 — Primary Sample Query (Assignment §3.1)
*The exact query from your assignment document.*

In [ ]:
response = run_rag_query(
    "Am I missing any must-watch videos for this athlete profile? "
    "Athlete: 19M, 200m Butterfly, underwater dolphin kick weakness, "
    "6 weeks to State Championships."
)

## 📋 Cell 11 — Additional Sample Queries (Assignment §3.2)

In [ ]:
# Query 2
run_rag_query("Which dolphin kick videos produced measurable improvement for male butterfly swimmers aged 18-22?")

In [ ]:
# Query 3
run_rag_query("What is the optimal video sequence for a butterfly swimmer 5-6 weeks out from a championship?")

In [ ]:
# Query 4
run_rag_query("Which videos are typically introduced too early and later revisited for this profile?")

## ✏️ Cell 12 — Your Custom Query

In [ ]:
# Change the query string below and run this cell
my_query = "What videos should I use for a 22-year-old female backstroke swimmer with weak catch mechanics?"
run_rag_query(my_query)

## 🚫 Cell 13 — Out-of-Domain Guard Test
*Confirms the system rejects non-swimming queries as per §4.4.*

In [ ]:
run_rag_query("What is the capital of France?")